In [ ]:
from steer_core.DataManager import DataManager

from steer_materials.CellMaterials.Base import CurrentCollectorMaterial, InsulationMaterial, SeparatorMaterial
from steer_materials.CellMaterials.Electrode import CathodeMaterial, Binder, ConductiveAdditive, AnodeMaterial

from steer_opencell_design.Components.CurrentCollectors.Punched import PunchedCurrentCollector
from steer_opencell_design.Formulations.ElectrodeFormulations import CathodeFormulation, AnodeFormulation
from steer_opencell_design.Components.Electrodes import Cathode, Anode
from steer_opencell_design.Components.Separators import Separator
from steer_opencell_design.Constructions.Layups.MonoLayers import ZFoldMonoLayer
from steer_opencell_design.Constructions.ElectrodeAssemblies.Stacks import ZFoldStack

from steer_opencell_design import __version__

import pandas as pd
from datetime import datetime as dt

In [ ]:
db = DataManager()

In [ ]:
db.remove_data('cells', "name == 'Vendor E NFPP'")

In [ ]:
# Get current collector materials from the database

current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
insulation_material = InsulationMaterial.from_database("Aluminium Oxide, 95%")
separator_material = SeparatorMaterial.from_database("Polyethylene")

In [ ]:
# Create the current collector object

cathode_current_collector = PunchedCurrentCollector(
    material=current_collector_material,
    width=166,
    height=184,
    thickness=13,
    tab_width=50,
    tab_height=20,
    tab_position=50,
    coated_tab_height=4,
    insulation_width=5
)

cathode_active_material = CathodeMaterial.from_database("NFPP")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=1.88,
    mass_loading=19.63,
    insulation_material=insulation_material,
    insulation_thickness=3
)

In [ ]:
# Create the anode object

anode_current_collector = PunchedCurrentCollector(
    material=current_collector_material,
    width=168,
    height=186,
    thickness=13,
    tab_width=50,
    tab_height=20,
    tab_position=118,
    coated_tab_height=4,
    insulation_width=5
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor A)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.1,
    mass_loading=8,
    insulation_material=insulation_material,
    insulation_thickness=3
)


In [ ]:
# make the layup

separator = Separator(
    material=separator_material,
    width=192,
    thickness=25
)

my_monolayer = ZFoldMonoLayer(
    anode=my_anode,
    cathode=my_cathode,
    separator=separator,
    name="Vendor E NFPP",
)


In [ ]:
# make the stack

my_stack = ZFoldStack(
    layup=my_monolayer,
    n_layers=40,
    additional_separator_wraps=6,
    name="Vendor E NFPP",
)

In [ ]:
print(f"Stack cost: {my_stack.cost:.2f} USD")
print(f"Stack mass: {my_stack.mass:.2f} g")

In [ ]:
my_stack.get_side_view(width=1600, height=500).show()

In [ ]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_stack.name],
    'object': [my_stack.serialize()],
    'form_factor': ['Prismatic'],
    'internal_construction': ['Stacked'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [ ]:
db.get_data(table_name='cells')